<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day03_practice3_%EB%B0%91%EB%B0%94%EB%8B%A5_%EC%8B%A0%EA%B2%BD%EB%A7%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [3]:
import numpy as np
np.random.seed(42)

# 셀 1. XOR 데이터
X_np = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=np.float64)
y_np = np.array([[0], [1], [1], [0]], dtype=np.float64)

In [4]:
# 셀 2. 넘파이 밑바닥 신경망 (심화학습2)
# 구조: 입력2 → 은닉8(시그모이드) → 출력1(시그모이드), 손실 MSE

def sigmoid(z): # 시그모이드 함수:
  return 1/(1 + np.exp(-z))

def sigmoid_prime(s): # 시그모이드 미분: σ' = σ(1-σ)
  return s * (1-s)

hidden = 8
W1 = np.random.randn(2, hidden) * 0.5     # (2,8) 입력2 → 은닉8
b1 = np.zeros((1, hidden))                # (1,8)
W2 = np.random.randn(hidden, 1) * 0.5     # (8,1) 은닉8 → 출력1
b2 = np.zeros((1,1))                      # (1,1)

lr = 0.5

In [7]:
# 셀 3. 넘파이로 직접 순전파, 역전파 실행
for epoch in range(10001):
  # 순전파
  z1 = X_np @ W1 + b1   # (4,8)
  h = sigmoid(z1)
  z2 = h @ W2 + b2      # (4,1)
  y_hat = sigmoid(z2)

  loss = ((y_hat - y_np) ** 2).mean()

  # 역전파: 체인 룰을 행렬로
  n = len(X_np)
  d_yhat = 2 * (y_hat - y_np) / n # ∂L/∂ŷ
  d_z2 = d_yhat * sigmoid_prime(y_hat) # σ'(z2) 시그모이드 미분을 곱해주기
  grad_W2 = h.T @ d_z2
  grad_b2 = d_z2.sum(axis=0, keepdims=True) # (n.1)를 행방향으로 위아래 더해서 하나의 값이 나오도록 (1.) -> 기존 2차원 형태(1.1)

  d_h = d_z2 @ W2.T
  d_z1 = d_h * sigmoid_prime(h)

  grad_W1 = X_np.T @ d_z1
  grad_b1 = d_z1.sum(axis=0, keepdims=True)

  # 갱신(경사하강)
  W1 -= lr * grad_W1; b1 -= lr * grad_b1
  W2 -= lr * grad_W2; b2 -= lr * grad_b2

  if epoch % 2000 == 0:
    print(f"[넘파이] epoch {epoch:5d} | loss {loss:.4f}")

pred_np = (y_hat > 0.5).astype(int).flatten()
print("넘파이 신경망 XOR 예측:", pred_np.tolist(), "| 정답: [0, 1, 1, 0]")
acc_np = (pred_np == y_np.flatten()).mean()
print(f"정확도: {acc_np:.0%}")

[넘파이] epoch     0 | loss 0.0006
[넘파이] epoch  2000 | loss 0.0005
[넘파이] epoch  4000 | loss 0.0004
[넘파이] epoch  6000 | loss 0.0003
[넘파이] epoch  8000 | loss 0.0003
[넘파이] epoch 10000 | loss 0.0002
넘파이 신경망 XOR 예측: [0, 1, 1, 0] | 정답: [0, 1, 1, 0]
정확도: 100%


In [10]:
# 셀 4. 같은 신경망을 PyTorch nn.Module로 ('정식 방법'인 클래스 작성법)

class XORNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer1 = nn.Linear(2,8)
    self.layer2 = nn.Linear(8,1)

  def forward(self, x):
    x = torch.sigmoid(self.layer1(x))
    x = torch.sigmoid(self.layer2(x))
    return x

model = XORNet()
X_t = torch.tensor(X_np, dtype=torch.float32) # PyTorch 텐서(Tensor)로 변환
y_t = torch.tensor(y_np, dtype=torch.float32)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

for epoch in range(10001):
  pred = model(X_t)           # 1 예측
  loss = loss_fn(pred, y_t)   # 2 손실
  optimizer.zero_grad()
  loss.backward()             # 3 역전파
  optimizer.step()            # 4 갱신
  if epoch % 2000 == 0:
    print(f"[PyTorch] epoch {epoch:5d} | loss {loss:.4f}")

with torch.no_grad():   # 평가는 no_grad
  pred_pt = (model(X_t) > 0.5).float().numpy().flatten() # 학습된 모델 model에 입력 데이터 X_t를 다시 넣어 예측값을 얻는다
print("PyTorch 신경망 XOR 예측:", pred_pt.tolist(), "| 정답: [0, 1, 1, 0]")

[PyTorch] epoch     0 | loss 0.2517
[PyTorch] epoch  2000 | loss 0.0262
[PyTorch] epoch  4000 | loss 0.0027
[PyTorch] epoch  6000 | loss 0.0012
[PyTorch] epoch  8000 | loss 0.0008
[PyTorch] epoch 10000 | loss 0.0006
PyTorch 신경망 XOR 예측: [0.0, 1.0, 1.0, 0.0] | 정답: [0, 1, 1, 0]
